# Pretraining the Model (Training Loop)

## Objectives
By the end of this demo you will be able to:
1. **Load** the initialised model checkpoint saved from Google Drive.
2. **Wrap** the packed Parquet dataset 3 into a PyTorch-compatible `Dataset` class.
3. **Configure** all training hyperparameters using HuggingFace `TrainingArguments`.
4. **Run** the pretraining loop using HuggingFace `Trainer` with a custom loss callback.
5. **Monitor** training loss to verify the model is learning.
6. **Evaluate** an intermediate checkpoint by running inference and comparing output quality.
7. **Save** the trained model to Google Drive.

## Description
This is the core of the pretraining pipeline — the training loop where the model
learns from token sequences by repeatedly predicting the next token and updating
its weights via backpropagation.

We use HuggingFace's `Trainer` API which handles the training loop, gradient
accumulation, mixed precision, checkpointing, and logging — all configurable
through a single `TrainingArguments` dataclass.

**Model used:** `SmolLM2-26L-pruned-init` (saved previuosly — depth-pruned SmolLM2-360M)
**Dataset used:** `packaged_pretrain_dataset.parquet` (packed token sequences)

> Pretraining is compute-intensive. This demo uses a small
> model (~300M params) and only 30 training steps — designed to run on a free
> Colab GPU in minutes. Real pretraining runs take days to months on hundreds
> of GPUs. Always estimate costs carefully before scaling up.


In [1]:
from google.colab import drive
drive.mount('/content/drive')

data_dir = "/content/drive/MyDrive/Colab Notebooks/data"
print(f"Data directory: {data_dir}")


Mounted at /content/drive
Data directory: /content/drive/MyDrive/Colab Notebooks/data


In [2]:
# Uncomment if not already installed
!pip install transformers torch datasets -q


In [3]:
import warnings
warnings.filterwarnings('ignore')

import torch

def fix_torch_seed(seed=42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

fix_torch_seed()
print(f"PyTorch version : {torch.__version__}")
print(f"Device available: {'GPU - ' + torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")


PyTorch version : 2.10.0+cpu
Device available: CPU


## 1. Load the Model to be Trained

We load the depth-pruned SmolLM2 checkpoint previously saved
This is our starting point — a model with pretrained weights and a modified
architecture ready for continued pretraining on
our custom corpus.

**Why `use_cache=False`?**
Gradient checkpointing (enabled in training args below) is **incompatible** with
KV-cache. Gradient checkpointing trades compute for memory — instead of storing
all intermediate activations during the forward pass, it recomputes them on-the-fly
during the backward pass. This significantly reduces GPU memory usage, allowing
training of larger models on the same hardware.

> Always set `use_cache=False` when `gradient_checkpointing=True`.
> Forgetting this is one of the most common errors when setting up a training run.


In [5]:
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained(
    f"{data_dir}/SmolLM2-26L-pruned-init",
    device_map="cpu",
    torch_dtype=torch.bfloat16,
    use_cache=False,
)

nparams = sum(p.numel() for p in model.parameters())
print(f"Model loaded successfully")
print(f"Architecture   : {model.config.model_type}")
print(f"Layers         : {model.config.num_hidden_layers}")
print(f"Hidden size    : {model.config.hidden_size}")
print(f"Total params   : {nparams:,} (~{nparams/1e6:.0f}M)")


Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

Model loaded successfully
Architecture   : llama
Layers         : 30
Hidden size    : 960
Total params   : 342,156,480 (~342M)


## 2. Load the Packed Dataset

We wrap the Parquet file in a custom PyTorch `Dataset` class.
HuggingFace's `Trainer` requires a `torch.utils.data.Dataset` object with
two methods:
- `__len__` → returns the total number of sequences
- `__getitem__` → returns one training sample as a dictionary

### Why `input_ids == labels`?

In **causal language modelling** (CLM), the training objective is next-token
prediction. The label for each token is simply the *next* token in the sequence.
The `Trainer` internally shifts labels by one position during loss computation,
so we pass the same tensor as both `input_ids` and `labels`.

> Token stream : [BOS] The cat sat on the mat [EOS]

> Input : [BOS] The cat sat on the mat

> Labels : The cat sat on the mat [EOS]




This training strategy is called **teacher forcing** — during training the model
always receives the ground-truth previous token as input, not its own prediction.
This stabilises training but creates a gap with inference (where the model uses
its own outputs) — known as **exposure bias**.


In [16]:
import datasets
from torch.utils.data import Dataset
import torch
import torch.nn.functional as F

class CustomDataset(Dataset):
    def __init__(self, args, split="train"):
        """Load the packed Parquet dataset and wrap it for PyTorch."""
        self.args = args
        self.max_seq_length = args.max_seq_length
        self.dataset = datasets.load_dataset(
            "parquet",
            data_files=args.dataset_name,
            split=split
        )

    def __len__(self):
        """Return total number of packed sequences."""
        return len(self.dataset)

    def __getitem__(self, idx):
        """
        Return one packed sequence padded/truncated to max_seq_length.
        Ensures all tensors in a batch are the same size.
        """
        input_ids = torch.LongTensor(self.dataset[idx]["input_ids"])

        # Truncate if longer than max_seq_length
        input_ids = input_ids[:self.max_seq_length]

        # Pad with 0 if shorter than max_seq_length
        pad_length = self.max_seq_length - len(input_ids)
        if pad_length > 0:
            input_ids = F.pad(input_ids, (0, pad_length), value=0)

        labels = input_ids.clone()

        return {"input_ids": input_ids, "labels": labels}


## 3. Configure Training Arguments

All hyperparameters are defined in a single `CustomArguments` dataclass that
extends `transformers.TrainingArguments`. This is the recommended HuggingFace
pattern — every run is fully reproducible and configurable from one place.

| Parameter | Value | What it does |
|---|---|---|
| `max_steps` | 30 | Total gradient update steps (keep small for demo) |
| `per_device_train_batch_size` | 2 | Sequences processed per GPU per step |
| `learning_rate` | 5e-5 | Step size for the AdamW optimiser |
| `warmup_steps` | 10 | LR ramps up linearly for first 10 steps, then decays |
| `lr_scheduler_type` | linear | LR decays linearly from peak to 0 after warmup |
| `weight_decay` | 0.01 | L2 regularisation — penalises large weights |
| `gradient_checkpointing` | True | Recompute activations on backward pass to save memory |
| `bf16` | True | bfloat16 mixed precision — halves memory, minimal accuracy loss |
| `gradient_accumulation_steps` | 1 | Effective batch = batch_size × accumulation_steps |
| `logging_steps` | 3 | Log loss every 3 steps |

> **LR warmup:** Without warmup, a large learning rate at step 0
> can cause the loss to diverge (explode) because the gradients are noisy and
> the optimiser hasn't yet found a stable direction. Warmup slowly ramps the LR
> from 0 to the target value, giving the model time to stabilise before taking
> large weight update steps.

> **Effective batch size:** With `per_device_train_batch_size=2` and
> `gradient_accumulation_steps=4`, the effective batch size = 8 sequences per step.
> Gradient accumulation simulates a larger batch without requiring more GPU memory.


In [25]:
import os

packed_path = f"{data_dir}/packaged_pretrain_dataset.parquet"
cleaned_path = f"{data_dir}/preprocessed_dataset.parquet"

print("File check:")
print(f"  packaged_pretrain_dataset.parquet : {'✅ EXISTS' if os.path.exists(packed_path) else '❌ NOT FOUND'}")
print(f"  preprocessed_dataset.parquet     : {'✅ EXISTS' if os.path.exists(cleaned_path) else '❌ NOT FOUND'}")

# List everything in data_dir
print(f"\nAll files in {data_dir}:")
for f in sorted(os.listdir(data_dir)):
    size_kb = os.path.getsize(os.path.join(data_dir, f)) / 1024
    print(f"  {f:<50} {size_kb:.1f} KB")


File check:
  packaged_pretrain_dataset.parquet : ✅ EXISTS
  preprocessed_dataset.parquet     : ✅ EXISTS

All files in /content/drive/MyDrive/Colab Notebooks/data:
  MNIST                                              4.0 KB
  SmolLM2-26L-pruned-init                            4.0 KB
  cifar-10-batches-py                                4.0 KB
  cifar-10-python.tar.gz                             166502.0 KB
  packaged_pretrain_dataset.parquet                  320.0 KB
  preprocessed_dataset.parquet                       1650.2 KB
  training_output                                    4.0 KB


In [18]:
from dataclasses import dataclass, field
import transformers

@dataclass
class CustomArguments(transformers.TrainingArguments):
    # Dataset
    dataset_name: str = field(
        default=f"{data_dir}/packaged_pretrain_dataset.parquet"
    )
    num_proc: int = field(default=1)            # subprocesses for data loading
    max_seq_length: int = field(default=32)     # must match Lab 3 packing length

    # Core training
    seed: int = field(default=42)
    optim: str = field(default="adamw_torch")   # AdamW — standard for LLM training
    max_steps: int = field(default=30)          # increase for real training runs
    per_device_train_batch_size: int = field(default=2)

    # Learning rate schedule
    learning_rate: float = field(default=5e-5)
    weight_decay: float = field(default=0.01)   # L2 regularisation
    warmup_steps: int = field(default=10)       # linear LR warmup
    lr_scheduler_type: str = field(default="linear")

    # Memory & precision
    gradient_checkpointing: bool = field(default=True)
    bf16: bool = field(default=True)            # requires Ampere GPU or newer
    gradient_accumulation_steps: int = field(default=1)
    dataloader_num_workers: int = field(default=2)

    # Logging
    logging_steps: int = field(default=3)
    report_to: str = field(default="none")      # set to "wandb" to enable tracking

    # Saving (uncomment to save intermediate checkpoints)
    # save_strategy: str = field(default="steps")
    # save_steps: int = field(default=10)
    # save_total_limit: int = field(default=2)


Parse the custom arguments and set the output directory where logs and
checkpoints will be saved:


In [19]:
#IF setup supports bf16/gpu
#parser = transformers.HfArgumentParser(CustomArguments)
#args, = parser.parse_args_into_dataclasses(
#    args=["--output_dir", f"{data_dir}/training_output"]
#)
#print(f"Output dir     : {args.output_dir}")
#print(f"Dataset        : {args.dataset_name}")
#print(f"Max steps      : {args.max_steps}")
#print(f"Batch size     : {args.per_device_train_batch_size}")
#print(f"Learning rate  : {args.learning_rate}")
#print(f"Warmup steps   : {args.warmup_steps}")
#print(f"Precision      : {'bfloat16' if args.bf16 else 'float32'}")

In [20]:
parser = transformers.HfArgumentParser(CustomArguments)
args, = parser.parse_args_into_dataclasses(
    args=["--output_dir", f"{data_dir}/training_output", "--bf16", "False"]
)
print(f"Output dir     : {args.output_dir}")
print(f"Dataset        : {args.dataset_name}")
print(f"Max steps      : {args.max_steps}")
print(f"Batch size     : {args.per_device_train_batch_size}")
print(f"Learning rate  : {args.learning_rate}")
print(f"Warmup steps   : {args.warmup_steps}")
print(f"Precision      : {'bfloat16' if args.bf16 else 'float32'}")

Output dir     : /content/drive/MyDrive/Colab Notebooks/data/training_output
Dataset        : /content/drive/MyDrive/Colab Notebooks/data/packaged_pretrain_dataset.parquet
Max steps      : 30
Batch size     : 2
Learning rate  : 5e-05
Warmup steps   : 10
Precision      : float32


Instantiate the training dataset and verify its shape matches
the `max_seq_length` set previously


In [21]:
train_dataset = CustomDataset(args=args)

print(f"Total packed sequences : {len(train_dataset):,}")
print(f"Shape of one sample    : {train_dataset[0]['input_ids'].shape}")
print(f"\nFirst 10 token IDs of sample[0]:")
print(train_dataset[0]['input_ids'][:10])


Total packed sequences : 442
Shape of one sample    : torch.Size([32])

First 10 token IDs of sample[0]:
tensor([50256,  2311,    73, 13090,   645,   569, 18354,  7496,   513,  1058])


## 4. Run the Trainer and Monitor Loss

### 4a. Loss Logging Callback

We define a custom `TrainerCallback` to capture the loss value at each
logging step. Callbacks in HuggingFace `Trainer` allow you to inject custom
logic at any point in the training loop — on step start/end, on log, on save, etc.

The training objective is to minimise the **cross-entropy loss** between the
model's predicted probability distribution over the vocabulary and the actual
next token:

Loss = −log P(correct next token)

- A randomly initialised model with vocab size 49,152 starts at: loss ≈ log(49152) ≈ **10.8**
- A well-pretrained model on general text achieves: loss ≈ **2–3**
- A decreasing loss curve confirms the model is learning

> If loss plateaus early or increases, common causes are:
> learning rate too high (use warmup or reduce LR), batch size too small,
> or corrupted/misformatted training data.


In [22]:
from transformers import Trainer, TrainerCallback

class LossLoggingCallback(TrainerCallback):
    """Captures training loss at each logging step for post-training analysis."""
    def __init__(self):
        self.logs = []

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is not None:
            self.logs.append(logs)

loss_logging_callback = LossLoggingCallback()
print("Loss logging callback initialised.")


Loss logging callback initialised.


### 4b. Run the Training Loop

The HuggingFace `Trainer` handles the complete training loop:

1. **Forward pass** → model predicts next token for every position
2. **Loss computation** → cross-entropy between prediction and ground truth
3. **Backward pass** → compute gradients via backpropagation
4. **Optimiser step** → update weights using AdamW
5. **LR scheduler step** → adjust learning rate according to schedule
6. Repeat for `max_steps` steps

All of this is encapsulated in a single `trainer.train()` call.


In [23]:
# Check if sequences are uniform length
lengths = [len(train_dataset.dataset[i]["input_ids"]) for i in range(min(20, len(train_dataset)))]
print(f"Sample sequence lengths (first 20): {lengths}")
print(f"Min length : {min(lengths)}")
print(f"Max length : {max(lengths)}")
print(f"max_seq_length set to: {args.max_seq_length}")


Sample sequence lengths (first 20): [168, 109, 114, 233, 326, 93, 227, 254, 242, 315, 280, 353, 289, 102, 80, 146, 161, 107, 82, 270]
Min length : 80
Max length : 353
max_seq_length set to: 32


In [24]:
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=None,
    callbacks=[loss_logging_callback]
)

print("Starting pretraining run...")
print(f"Training for {args.max_steps} steps on {len(train_dataset):,} sequences\n")
trainer.train()


Starting pretraining run...
Training for 30 steps on 442 sequences



IndexError: index out of range in self

### 4c. Inspect the Loss Values

Print the captured loss values step by step.
A consistently **decreasing** trend confirms correct training behaviour:


In [ ]:
print(f"{'Step':<10} {'Loss':<10}")
print("-" * 20)
for log in loss_logging_callback.logs:
    if 'loss' in log:
        step = log.get('step', log.get('epoch', '?'))
        print(f"{str(step):<10} {log['loss']:<10.4f}")


### 4d. Plot the Training Curve

Visualise the loss curve to observe the rate and stability of learning.
Save the plot to Google Drive for records:
